# 00 — Understand the Taobao data

**Goal of this step:** look at the 3 raw tables before building anything on top of them.

**Analogy:** a shop window.
- `raw_sample` = the camera counting every person who walked past each poster, and whether they stopped (clicked).
- `ad_feature` = the label on the back of each poster (product, campaign, advertiser).
- `user_profile` = the loyalty-card info about the passers-by (gender, age, spending level).

8 days of data: **6–13 May 2017**, Beijing time. "Now" in our project = 13 May 23:59.

In [1]:
import duckdb

con = duckdb.connect("../data/ads.duckdb", read_only=True)

def show(sql):
    return con.sql(sql).df()   # run SQL, show the result as a table

## 1. `raw_sample` — the event log
One row = **one ad shown to one user** (an impression). `clk` = 1 if they clicked, `nonclk` = 1 if not.
`pid` = the placement (which slot on the page).

In [2]:
show("select * from raw_sample limit 5")

,user_id,time_stamp,adgroup_id,pid,nonclk,clk
0,581738,1494137644,1,430548_1007,1,0
1,449818,1494638778,3,430548_1007,1,0
2,914836,1494650879,4,430548_1007,1,0
3,914836,1494651029,5,430548_1007,1,0
4,399907,1494302958,8,430548_1007,1,0


How big is it, and how many users and ads?

In [3]:
show("""
select count(*)                   as impressions,
       sum(clk)                   as clicks,
       round(avg(clk) * 100, 2)   as ctr_pct,
       count(distinct user_id)    as users,
       count(distinct adgroup_id) as ads
from raw_sample
""")

,impressions,clicks,ctr_pct,users,ads
0,26557961,1366056.0,5.14,1141729,846811


Check: every row is **either** a click **or** a non-click (so impressions = number of rows). Expect 0.

In [4]:
show("select count(*) as bad_rows from raw_sample where clk + nonclk <> 1")

,bad_rows
0,0


### Per day
`time_stamp` is in Unix seconds, so we convert it to Beijing time first.
CTR is very steady (~5%). Notice **12 May** is the lowest day — a nice "what happened?" question.

In [5]:
show("""
select strftime(to_timestamp(time_stamp) at time zone 'Asia/Shanghai', '%Y-%m-%d') as day,
       count(*)                 as impressions,
       sum(clk)                 as clicks,
       round(avg(clk) * 100, 2) as ctr_pct
from raw_sample
group by day
order by day
""")

,day,impressions,clicks,ctr_pct
0,2017-05-06,3270348,170332.0,5.21
1,2017-05-07,3430002,175780.0,5.12
2,2017-05-08,3354523,174394.0,5.20
3,2017-05-09,3248016,169606.0,5.22
4,2017-05-10,3333752,172190.0,5.17
5,2017-05-11,3378604,177827.0,5.26
6,2017-05-12,3234051,159437.0,4.93
7,2017-05-13,3308665,166490.0,5.03


### Per placement — only 2 slots on the page

In [6]:
show("""
select pid,
       count(*)                 as impressions,
       round(avg(clk) * 100, 2) as ctr_pct
from raw_sample
group by pid
""")

,pid,impressions,ctr_pct
0,430539_1007,10085063,5.36
1,430548_1007,16472898,5.01


## 2. `ad_feature` — what each ad is
Family tree: **advertiser (`customer`) → campaign → ad (`adgroup_id`)**, plus category, brand and product price.

In [7]:
show("select * from ad_feature limit 5")

,adgroup_id,cate_id,campaign_id,customer,brand,price
0,63133,6406,83237,1,95471,170.00
1,313401,6406,83237,1,87331,199.00
2,248909,392,83237,1,32233,38.00
3,208458,392,83237,1,174374,139.00
4,110847,7211,135256,2,145952,32.99


In [8]:
show("""
select count(*)                    as ads,
       count(distinct campaign_id) as campaigns,
       count(distinct customer)    as advertisers,
       count(distinct cate_id)     as categories,
       count(distinct brand)       as brands
from ad_feature
""")

,ads,campaigns,advertisers,categories,brands
0,846811,423436,255875,6769,99814


**Gotcha 1:** about 29% of ads have no brand. "By brand" questions silently drop them.

In [9]:
show("select count(*) as ads, sum(brand is null) as no_brand from ad_feature")

,ads,no_brand
0,846811,246330.0


**Gotcha 2:** price has junk outliers (max 99,999,999). The average is misleading — use the median.

In [10]:
show("""
select round(avg(price))          as avg_price,
       median(price)              as median_price,
       max(price)                 as max_price,
       sum(price > 100000)        as ads_over_100k
from ad_feature
""")

,avg_price,median_price,max_price,ads_over_100k
0,1839.0,139.0,99999999.0,321.0


Good news: every ad in the log has a row in `ad_feature`, so this join is safe. Expect 0.

In [11]:
show("""
select count(*) as log_rows_without_ad
from raw_sample r
left join ad_feature a using (adgroup_id)
where a.adgroup_id is null
""")

,log_rows_without_ad
0,0


## 3. `user_profile` — who the shoppers are
| column | meaning |
|---|---|
| `final_gender_code` | 1 = male, 2 = female |
| `age_level` | age bucket 0–6 |
| `pvalue_level` | spending power: 1 low, 2 mid, 3 high |
| `shopping_level` | 1 shallow, 2 medium, 3 deep shopper |
| `occupation` | 1 = college student |
| `new_user_class_level` | city tier 1–4 |
| `cms_segid`, `cms_group_id` | Taobao's internal segments |

In [12]:
show("select * from user_profile limit 5")

,user_id,cms_segid,cms_group_id,final_gender_code,age_level,pvalue_level,shopping_level,occupation,new_user_class_level
0,234,0,5,2,5,<NA>,3,0,3
1,523,5,2,2,2,1,3,1,2
2,612,0,8,1,2,2,3,0,<NA>
3,1670,0,4,2,4,<NA>,1,0,<NA>
4,2545,0,10,1,4,<NA>,3,0,<NA>


In [13]:
show("select final_gender_code, count(*) as users from user_profile group by 1 order by 1")

,final_gender_code,users
0,1,377517
1,2,684251


In [14]:
show("select age_level, count(*) as users from user_profile group by 1 order by 1")

,age_level,users
0,0,501
1,1,65568
2,2,189617
3,3,307340
4,4,261751
5,5,214462
6,6,22529


**Gotcha 3:** many blanks. Spending power is missing for more than half of users.

In [13]:
show("""
select count(*)                             as users,
       sum(pvalue_level is null)            as no_spending_power,
       sum(new_user_class_level is null)    as no_city_tier
from user_profile
""")

,users,no_spending_power,no_city_tier
0,1061768,575917.0,344920.0


**Gotcha 4 (important for the agent):** some log rows belong to users with **no profile**.
An inner join to `user_profile` quietly loses them, so "by audience" totals come out lower than overall totals.

In [14]:
show("""
select count(*) as log_rows_without_profile,
       round(count(*) * 100.0 / (select count(*) from raw_sample), 1) as pct_of_log
from raw_sample r
left join user_profile u using (user_id)
where u.user_id is null
""")

,log_rows_without_profile,pct_of_log
0,1528526,5.8


## 4. What is *not* in the data
No cost, no spend, no purchases. Only **impressions and clicks**.
So our metrics are: **impressions, clicks, CTR, reach (unique users)**.

## 5. How this becomes our 28 tables (M1)
```
raw_sample   -> fact_ad_events  (adds Beijing time, date, hour)
ad_feature   -> dim_ad, dim_campaign, dim_advertiser
user_profile -> dim_user
pid          -> dim_placement
    |
agg_*            daily / hourly / weekly totals
*_valid          suspicious clickers removed
reach_*          unique users
live_*_last6h    today's latest 6 hours
```
Many of these tables *look alike* — picking the right one for a question is the problem our agent solves.

In [15]:
con.close()